In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from keras.models import Model
from keras.layers import Dense, Input, Dropout, LSTM, Activation
from keras.layers.embeddings import Embedding
from keras.preprocessing import sequence
from keras.initializers import glorot_uniform

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# https://www.kaggle.com/competitions/tweet-sentiment-extraction/overview
# 3 emotions, approx 30k labelled tweets
df_train = pd.read_csv("/content/train.csv",encoding='ISO-8859-1',header=None)

In [ ]:
df_train.drop(columns=[2,0],inplace=True)

In [ ]:
df_test = pd.read_csv("/content/test.csv",encoding='ISO-8859-1',header=None)

In [ ]:
df_test.drop(columns=[0],inplace=True)

In [ ]:
df_test.drop([df_test.index[0]],inplace=True)
df_train.drop([df_train.index[0]],inplace=True)

In [ ]:
df_test.columns = ['text','Sentiment']
df_train.columns = ['text','Sentiment']

In [ ]:
df_train.dropna(axis=0,inplace=True)
df_test.dropna(axis=0,inplace=True)

In [ ]:
# Function to Clean the Tweet.
import re
def clean_tweet(tweet):
    return ' '.join(re.sub('(\\\\n)|(b\"[^0-9A-Za-z A-Za-z0-9 \t]+)|(b\'[^0-9A-Za-z]+)|(b\"[A-Za-z0-9]+)|(b\'[A-Za-z0-9]+)|(b\'#[A-Za-z0-9]+)|(b\'@[A-Za-z0-9]+)|(\\\\x[A-Za-z0-9]+)|(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)|([RT])', ' ', str(tweet).lower()).split())


In [ ]:
# Call function to get Clean tweets
df_test["CleanTweet"] = df_test['text'].apply(lambda x : clean_tweet(x))
df_train["CleanTweet"] = df_train['text'].apply(lambda x : clean_tweet(x))

In [ ]:
df_test.drop(df_test.index[df_test.CleanTweet.eq("")], inplace=True)
df_train.drop(df_train.index[df_train.CleanTweet.eq("")], inplace=True)

In [ ]:
#Emotion dictionary lookup
emotions_dict = {"negative":0, "neutral":2, "positive":4}
emotions_dict

def emo_lookup(emo):
  return emotions_dict[emo]

df_test['Label'] = df_test.Sentiment.apply(emo_lookup)
df_train['Label'] = df_train.Sentiment.apply(emo_lookup)

In [ ]:
X_train = df_train.CleanTweet
X_test = df_test.CleanTweet
Y_train = df_train.Label
Y_test = df_test.Label

In [ ]:
def read_glove_vecs(glove_file):
    with open(glove_file, 'r',encoding='UTF-8') as f:
        words = set()
        word_to_vec_map = {}
        for line in f:
            line = line.strip().split()
            curr_word = line[0]
            words.add(curr_word)
            word_to_vec_map[curr_word] = np.array(line[1:], dtype=np.float64)
        
        i = 1
        words_to_index = {}
        index_to_words = {}
        for w in sorted(words):
            words_to_index[w] = i
            index_to_words[i] = w
            i = i + 1
    return words_to_index, index_to_words, word_to_vec_map

In [ ]:
np.asarray(X_train.values)
X_train.values

array(['i d have responded if i were going',
       'sooo sad i will miss you here in san diego',
       'my boss is bullying me', ...,
       'yay good for both of you enjoy the break you probably need it after such hectic weekend take care hun xxxx',
       'but it was worth it',
       'all this flirting going on the atg smiles yay hugs'], dtype=object)

In [ ]:
def sentences_to_indices(X, word_to_index, max_len):
    """
    Converts an array of sentences (strings) into an array of indices corresponding to words in the sentences.
    The output shape should be such that it can be given to `Embedding()` (described in Figure 4). 
 
    Arguments:
    X -- array of sentences (strings), of shape (m, 1)
    word_to_index -- a dictionary containing the each word mapped to its index
    max_len -- maximum number of words in a sentence. You can assume every sentence in X is no longer than this. 
    
    Returns:
    X_indices -- array of indices corresponding to words in the sentences from X, of shape (m, max_len)
    """

    m = X.shape[0]                                   # number of training examples

    X_indices =  np.zeros((m, max_len))
    
    for i in range(m):                               # loop over training examples

        # Convert the ith training sentence in lower case and split is into words. You should get a list of words.
        sentence_words = (X[i,].lower()).split()

        # Initialize j to 0
        j = 0

        # Loop over the words of sentence_words
        for w in sentence_words:
            # Set the (i,j)th entry of X_indices to the index of the correct word.
            X_indices[i,j] = word_to_index[w]
            # Increment j to j + 1
            j = j+1

    return X_indices

In [ ]:
maxLen = len(max(X_train, key=len).split())

index = 1
print(X_train[index], Y_train[index])

# SS - We can if needed
#Y_oh_train = convert_to_one_hot(Y_train, C = 5)
#Y_oh_test = convert_to_one_hot(Y_test, C = 5)

Y_oh_train = Y_train
Y_oh_test = Y_test

index = 50
print(Y_train[index], "is converted into one hot", Y_oh_train[index])

## SS - Difference between 50, 100, 200, 300D in glove vectors
word_to_index, index_to_word, word_to_vec_map = read_glove_vecs('/content/glove.6B.50d.txt')

i d have responded if i were going 2
0 is converted into one hot 0


In [ ]:
word = "cucumber"
index = 289846
print("the index of", word, "in the vocabulary is", word_to_index[word])
print("the", str(index) + "th word in the vocabulary is", index_to_word[index])
#the index of cucumber in the vocabulary is 113317
#the 289846th word in the vocabulary is potatos

X1 = np.array(["funny lol", "lets play baseball", "food is ready for you"])
X1_indices = sentences_to_indices(X1,word_to_index, max_len = 5)
print("X1 =", X1)
print("X1_indices =", X1_indices)

#X1 =	['funny lol' 'lets play baseball' 'food is ready for you']
#X1_indices =	[[ 155345. 225122. 0. 0. 0.]
#[ 220930. 286375. 69714. 0. 0.]
#[ 151204. 192973. 302254. 151349. 394475.]]

the index of cucumber in the vocabulary is 113317
the 289846th word in the vocabulary is potatos
X1 = ['funny lol' 'lets play baseball' 'food is ready for you']
X1_indices = [[155345. 225122.      0.      0.      0.]
 [220930. 286375.  69714.      0.      0.]
 [151204. 192973. 302254. 151349. 394475.]]


In [ ]:
X_train

1                       i d have responded if i were going
2               sooo sad i will miss you here in san diego
3                                   my boss is bullying me
4                            what interview leave me alone
5        sons of why couldn t they put them on the rele...
                               ...                        
27477    wish we could come see u on denver husband los...
27478    i ve wondered about rake to the client has mad...
27479    yay good for both of you enjoy the break you p...
27480                                  but it was worth it
27481    all this flirting going on the atg smiles yay ...
Name: CleanTweet, Length: 27478, dtype: object

In [ ]:
X1

array(['funny lol', 'lets play baseball', 'food is ready for you'],
      dtype='<U21')

In [ ]:
np.array(X_train)

array(['i d have responded if i were going',
       'sooo sad i will miss you here in san diego',
       'my boss is bullying me', ...,
       'yay good for both of you enjoy the break you probably need it after such hectic weekend take care hun xxxx',
       'but it was worth it',
       'all this flirting going on the atg smiles yay hugs'], dtype=object)

In [ ]:
X_train.shape

(27478,)

In [ ]:
X1.shape

(3,)

In [ ]:
def pretrained_embedding_layer(word_to_vec_map, word_to_index):
    """
    Creates a Keras Embedding() layer and loads in pre-trained GloVe 50-dimensional vectors.
    
    Arguments:
    word_to_vec_map -- dictionary mapping words to their GloVe vector representation.
    word_to_index -- dictionary mapping from words to their indices in the vocabulary (400,001 words)

    Returns:
    embedding_layer -- pretrained layer Keras instance
    """
    
    vocab_len = len(word_to_index) + 1                  # adding 1 to fit Keras embedding (requirement)
    emb_dim = word_to_vec_map["cucumber"].shape[0]      # define dimensionality of your GloVe word vectors (= 50)

    ### START CODE HERE ###
    # Initialize the embedding matrix as a numpy array of zeros of shape (vocab_len, dimensions of word vectors = emb_dim)
    emb_matrix = np.zeros([vocab_len,emb_dim])

    # Set each row "index" of the embedding matrix to be the word vector representation of the "index"th word of the vocabulary
    for word, index in word_to_index.items():
        emb_matrix[index, :] = word_to_vec_map[word]

    # Define Keras embedding layer with the correct output/input sizes, make it non-trainable. Use Embedding(...). Make sure to set trainable=False. 
    embedding_layer = Embedding(vocab_len, emb_dim, trainable=False)
    ### END CODE HERE ###

    # Build the embedding layer, it is required before setting the weights of the embedding layer. Do not modify the "None".
    embedding_layer.build((None,))

    # Set the weights of the embedding layer to the embedding matrix. Your layer is now pretrained.
    embedding_layer.set_weights([emb_matrix])
    
    return embedding_layer

embedding_layer = pretrained_embedding_layer(word_to_vec_map, word_to_index)
print("weights[0][1][3] =", embedding_layer.get_weights()[0][1][3])

#weights[0][1][3] =	-0.3403

weights[0][1][3] = -0.3403


In [ ]:
def Emojify_V2(input_shape, word_to_vec_map, word_to_index):
    """
    Function creating the Emojify-v2 model's graph.
    
    Arguments:
    input_shape -- shape of the input, usually (max_len,)
    word_to_vec_map -- dictionary mapping every word in a vocabulary into its 50-dimensional vector representation
    word_to_index -- dictionary mapping from words to their indices in the vocabulary (400,001 words)

    Returns:
    model -- a model instance in Keras
    """

    # Define sentence_indices as the input of the graph, it should be of shape input_shape and dtype 'int32' (as it contains indices).
    sentence_indices = Input(shape=input_shape,dtype='int32')
    
    embedding_layer = pretrained_embedding_layer(word_to_vec_map, word_to_index)
    
    # Propagate sentence_indices through your embedding layer, you get back the embeddings
    embeddings = embedding_layer(sentence_indices)   
    
    # Propagate the embeddings through an LSTM layer with 128-dimensional hidden state
    # Be careful, the returned output should be a batch of sequences.
    X = LSTM(128, return_sequences=True)(embeddings)
    # Add dropout with a probability of 0.5
    X = Dropout(0.5)(X)
    # Propagate X trough another LSTM layer with 128-dimensional hidden state
    # Be careful, the returned output should be a single hidden state, not a batch of sequences.
    X = LSTM(128, return_sequences=False)(embeddings)
    # Add dropout with a probability of 0.5
    X = Dropout(0.5)(X)
    # Propagate X through a Dense layer with softmax activation to get back a batch of 3-dimensional vectors.
    X = Dense(3, activation=None)(X)
    # Add a softmax activation
    X = Activation('softmax')(X)
    
    # Create Model instance which converts sentence_indices into X.
    model = Model(inputs=[sentence_indices], outputs=X)
    
    return model


In [ ]:
model = Emojify_V2((maxLen,), word_to_vec_map, word_to_index)
model.summary()

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

X_train_indices = sentences_to_indices(np.array(X_train), word_to_index, maxLen)
#np.asarray(X_train.values)

#Y_train_oh = convert_to_one_hot(Y_train, C = 5)
Y_train_oh = Y_train

model.fit(X_train_indices, Y_train_oh, epochs = 50, batch_size = 32, shuffle=True)

X_test_indices = sentences_to_indices(X_test, word_to_index, max_len = maxLen)

# SS If needed
#Y_test_oh = convert_to_one_hot(Y_test, C = 5)
Y_test_oh = Y_test

loss, acc = model.evaluate(X_test_indices, Y_test_oh)
print()
print("Test accuracy = ", acc)
#Test accuracy =  0.875000008515

# Change the sentence below to see your prediction. Make sure all the words are in the Glove embeddings.  
x_test = np.array(['jet lag'])
X_test_indices = sentences_to_indices(x_test, word_to_index, maxLen)
print(x_test[0] +' '+  np.argmax(model.predict(X_test_indices)))

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 29)]              0         
                                                                 
 embedding_4 (Embedding)     (None, 29, 50)            20000050  
                                                                 
 lstm_5 (LSTM)               (None, 128)               91648     
                                                                 
 dropout_5 (Dropout)         (None, 128)               0         
                                                                 
 dense_2 (Dense)             (None, 3)                 387       
                                                                 
 activation_2 (Activation)   (None, 3)                 0         
                                                                 
Total params: 20,092,085
Trainable params: 92,035
Non-train

KeyError: ignored